SetUp & Data Loading

In [1]:
!pip install datasets[audio] librosa scikit-learn hmmlearn jiwer evaluate

from datasets import load_dataset, DatasetDict, Audio
import numpy as np
import librosa

# Load same dataset
dataset = load_dataset(
    "google/WaxalNLP",
    "aka_asr",
    split=["train", "validation", "test"],
    verification_mode="no_checks"
)
dataset = DatasetDict({
    "train": dataset[0],
    "validation": dataset[1],
    "test": dataset[2]
})

# Resample to 16kHz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

 Feature Extraction (MFCCs — replaces Whisper's log-mel spectrogram)

In [35]:
import numpy as np
import librosa


def extract_features(audio_array, sr=16000, n_fft=400, hop_length=160, n_mels=80):

    audio = audio_array.astype(np.float32)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels
    )

    log_mel = librosa.power_to_db(mel, ref=np.max)

    delta = librosa.feature.delta(log_mel)
    delta2 = librosa.feature.delta(log_mel, order=2)

    features = np.vstack([log_mel, delta, delta2]).T

    # reduce memory usage
    features = features.astype(np.float16)

    return features


def prepare_features(split, max_samples=None):

    features_list = []
    labels_list = []

    data = dataset[split]

    if max_samples is not None:
        data = data.select(range(max_samples))

    for i, sample in enumerate(data):

        if i % 200 == 0:
            print(f"Processing {i}/{len(data)}")

        audio = sample["audio"]["array"]
        transcription = sample["transcription"]

        feats = extract_features(audio)

        features_list.append(feats)
        labels_list.append(transcription)

    return features_list, labels_list


# limit dataset size to prevent RAM crash
train_feats, train_labels = prepare_features("train", max_samples=2000)
test_feats, test_labels   = prepare_features("test", max_samples=500)

Processing 0/2000
Processing 200/2000
Processing 400/2000
Processing 600/2000
Processing 800/2000
Processing 1000/2000
Processing 1200/2000
Processing 1400/2000
Processing 1600/2000
Processing 1800/2000
Processing 0/500
Processing 200/500
Processing 400/500


In [3]:
!pip install hmmlearn

 Acoustic Model — GMM-HMM (the classical standard)

In [36]:
import numpy as np
import time
import warnings
from hmmlearn import hmm

warnings.filterwarnings("ignore", category=RuntimeWarning)

MAX_SEGMENTS_PER_CHAR = 500
MIN_SEGMENTS_PER_CHAR = 20


def build_char_hmm_models(
        transcriptions,
        features_list,
        n_components=2,
        n_mix=1,
        n_iter=20):

    print("Step 1/3: Collecting frames per character...")

    char_frames = {}

    for idx, (feats, text) in enumerate(zip(features_list, transcriptions)):

        if idx % 100 == 0:
            print(f"Processed {idx}/{len(features_list)} utterances", end="\r")

        T = len(feats)

        # remove spaces from training targets
        chars = [c for c in text.lower() if c.isalpha() or c == " "]

        if not chars:
            continue

        for ch in chars:

            seg_length = np.random.randint(10, 40)

            if T <= seg_length:
                continue

            start = np.random.randint(0, T - seg_length)
            end = start + seg_length

            segment = feats[start:end]

            char_frames.setdefault(ch, []).append(segment)

    print("\nCharacters found:", len(char_frames))
    print("\nStep 2/3: Training HMM models")

    models = {}
    start_time = time.time()

    for i, (char, segments) in enumerate(char_frames.items()):

        if len(segments) < MIN_SEGMENTS_PER_CHAR:
            print(f"Skipping '{char}' (too few samples: {len(segments)})")
            continue

        if len(segments) > MAX_SEGMENTS_PER_CHAR:
            segments = segments[:MAX_SEGMENTS_PER_CHAR]

        X = np.vstack(segments).astype(np.float32)

        # feature normalization
        mean = X.mean(axis=0)

        # variance floor to avoid degenerate Gaussians
        std = np.maximum(X.std(axis=0), 0.1)

        X = (X - mean) / std

        lengths = [len(s) for s in segments]

        print(f"[{i+1}/{len(char_frames)}] Training '{char}' with {len(segments)} segments")

        model = hmm.GMMHMM(
            n_components=n_components,
            n_mix=n_mix,
            covariance_type="diag",
            n_iter=n_iter,
            verbose=False
        )

        try:
            model.fit(X, lengths)
            models[char] = (model, mean, std)

        except Exception as e:
            print(f"Skipping '{char}' due to training error:", e)

    elapsed = time.time() - start_time

    print("\nFinished training")
    print("Total trained character models:", len(models))
    print(f"Training time: {elapsed:.1f} seconds")

    return models


print("=" * 60)
print("Training GMM-HMM models per character...")
print("=" * 60)

hmm_models = build_char_hmm_models(
    train_labels,
    train_feats,
    n_components=3,
    n_mix=2,
    n_iter=20
)

print("\nTraining complete.")

Training GMM-HMM models per character...
Step 1/3: Collecting frames per character...
Processed 1900/2000 utterances
Characters found: 31

Step 2/3: Training HMM models
[1/31] Training 'g' with 500 segments


[2/31] Training 'ɛ' with 500 segments
[3/31] Training 'l' with 500 segments


[4/31] Training 's' with 500 segments


[5/31] Training ' ' with 500 segments
[6/31] Training 'f' with 500 segments
[7/31] Training 'ɔ' with 500 segments
[8/31] Training 'r' with 500 segments
[9/31] Training 'e' with 500 segments


[10/31] Training 'y' with 500 segments
[11/31] Training 'i' with 500 segments
[12/31] Training 'n' with 500 segments
[13/31] Training 'a' with 500 segments
[14/31] Training 'h' with 500 segments
[15/31] Training 'o' with 500 segments
[16/31] Training 'b' with 500 segments
[17/31] Training 'k' with 500 segments
[18/31] Training 't' with 500 segments


[19/31] Training 'u' with 500 segments
[20/31] Training 'm' with 500 segments
[21/31] Training 'p' with 500 segments
[22/31] Training 'w' with 500 segments
[23/31] Training 'd' with 500 segments
[24/31] Training 'z' with 500 segments


[25/31] Training 'v' with 83 segments


[26/31] Training 'c' with 485 segments


Skipping 'x' (too few samples: 15)
Skipping 'ℇ' (too few samples: 12)
Skipping 'j' (too few samples: 12)
[30/31] Training 'ↄ' with 23 segments


Skipping 'q' (too few samples: 2)

Finished training
Total trained character models: 27
Training time: 731.9 seconds

Training complete.


In [37]:
def decode_with_hmm(features, models,
                    ngram_counts,
                    context_counts,
                    vocab,
                    chunk_size=40,
                    lm_weight=0.6):

    predicted = []
    chars = list(models.keys())

    T = len(features)

    for start in range(0, T, chunk_size):

        chunk = features[start:start + chunk_size]

        if len(chunk) < 5:
            continue

        best_char = None
        best_score = -np.inf

        for ch in chars:

            model, mean, std = models[ch]

            X = (chunk - mean) / std
            X = np.clip(X, -5, 5).astype(np.float32)

            try:
                acoustic_score = model.score(X)

                candidate = "".join(predicted + [ch])

                lm_score = ngram_score(
                    candidate,
                    ngram_counts,
                    context_counts,
                    vocab   # ← required argument
                )

                total_score = acoustic_score + lm_weight * lm_score

                if total_score > best_score:
                    best_score = total_score
                    best_char = ch

            except Exception as e:
                print("Decoder error:", e)
                continue

        if best_char is not None:
            predicted.append(best_char)

    return "".join(predicted)

Step 3B: Acoustic Model — SVM / Random Forest on MFCC statistics

In [38]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np


def utterance_features(feats):
    """
    Convert variable-length acoustic sequence into a fixed-size vector.
    """

    feats = feats.astype(np.float32)

    mean = feats.mean(axis=0)

    # variance floor
    std = np.maximum(feats.std(axis=0), 1e-5)

    return np.concatenate([mean, std])


# Build dataset
X_train = np.array([utterance_features(f) for f in train_feats], dtype=np.float32)
X_test  = np.array([utterance_features(f) for f in test_feats], dtype=np.float32)


# Predict first character baseline
y_train = [t[0] if len(t) > 0 else "?" for t in train_labels]
y_test  = [t[0] if len(t) > 0 else "?" for t in test_labels]


svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="rbf",
        C=10,
        gamma="scale"
    ))
])


print("Training SVM baseline...")
svm_pipeline.fit(X_train, y_train)

acc = svm_pipeline.score(X_test, y_test)

print(f"SVM first-character accuracy: {acc*100:.2f}%")

Training SVM baseline...
SVM first-character accuracy: 32.00%


Step 4: Language Model — n-gram (replaces Whisper's decoder)

In [39]:
from collections import defaultdict
import math


def build_ngram_lm(transcriptions, n=3):

    """
    Train character-level n-gram language model.
    """

    ngram_counts = defaultdict(lambda: defaultdict(int))
    context_counts = defaultdict(int)

    vocab = set()

    for text in transcriptions:

        text = text.lower().strip()

        padded = " " * (n - 1) + text + " "

        for i in range(len(padded) - n + 1):

            context = padded[i:i + n - 1]
            next_ch = padded[i + n - 1]

            ngram_counts[context][next_ch] += 1
            context_counts[context] += 1

            vocab.add(next_ch)

    vocab = sorted(vocab)

    return ngram_counts, context_counts, vocab


def ngram_score(text, ngram_counts, context_counts, vocab, n=3, alpha=0.1):

    """
    Compute log probability of a string under the n-gram LM.
    """

    text = text.lower().strip()

    # pad start and end
    padded = " " * (n - 1) + text + " "

    log_prob = 0.0
    V = len(vocab)

    for i in range(len(padded) - n + 1):

        context = padded[i:i + n - 1]
        ch = padded[i + n - 1]

        count = ngram_counts[context].get(ch, 0)
        total = context_counts.get(context, 0)

        prob = (count + alpha) / (total + alpha * V)

        log_prob += math.log(prob + 1e-12)

    return log_prob


print("Building character 3-gram language model...")

ngram_counts, context_counts, vocab = build_ngram_lm(train_labels, n=3)

print("Vocabulary size:", len(vocab))
print("Unique contexts:", len(ngram_counts))

Building character 3-gram language model...
Vocabulary size: 45
Unique contexts: 680


In [8]:
!pip install evaluate

In [9]:
!pip install jiwer

In [10]:
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print("Test size:", len(dataset["test"]))

Train size: 10107
Validation size: 1123
Test size: 1522


In [11]:
print(len(train_feats))
print(len(test_feats))
print(train_feats[0].shape)

2000
500
(1516, 240)


Step 5: Evaluation — WER & CER (same metrics as your Whisper code)

In [40]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def normalize_text(text):
    """
    Normalize text for fair comparison.
    """
    text = text.lower().strip()
    text = " ".join(text.split())
    return text


def evaluate_classical(features_list, references, models, max_samples=None):

    predictions = []
    refs_clean = []

    if max_samples is None:
        max_samples = len(features_list)

    for i, (feats, ref) in enumerate(zip(features_list[:max_samples],
                                         references[:max_samples])):

        # Decode prediction
        pred = decode_with_hmm(
        feats,
        models,
        ngram_counts,
        context_counts,
        vocab
)

        # Normalize
        pred = normalize_text(pred)
        ref = normalize_text(ref)

        predictions.append(pred)
        refs_clean.append(ref)

        if i % 20 == 0:
            print(f"{i}/{max_samples}")
            print("REF :", ref[:50])
            print("PRED:", pred[:50])
            print()

    # Compute metrics
    wer = wer_metric.compute(
        predictions=predictions,
        references=refs_clean
    )

    cer = cer_metric.compute(
        predictions=predictions,
        references=refs_clean
    )

    print("\nModel Results")
    print("-------------------------------")
    print(f"WER: {wer*100:.2f}%")
    print(f"CER: {cer*100:.2f}%")

    return wer, cer


# Run evaluation
wer, cer = evaluate_classical(
    test_feats,
    test_labels,
    hmm_models,
    max_samples=50
)

0/50
REF : mbenyinfo nson gyina agoprama no so. ana wɔ afa ɛn
PRED: kffkhkkkkfkbkrkdhkkkkkddhhdbkwkdkhddfkkɛffdfdkhhyk



20/50
REF : nnipa baanu te ɔkoro so. ↄkoro no kɛntɛn si mu. ↄb
PRED: mmmmmhbbhmmhɛhmmmhhɛhbhmmɛɛɛɛhbhhmmbhhmmhhbbbmmmhb



40/50
REF : nnipa nson gyina biribi so a mmarima nnan asɔ ɔbaa
PRED: krmafhdbɛddbdakbyhdɛkffyhddiaddhdkdhhbhɛɛdfddkdrkh




Model Results
-------------------------------
WER: 100.00%
CER: 90.47%
